# 02 · Address quality: linking KBO addresses to BeST

Notebook 01 built a `building_key` (postcode + normalised street + house number) for every address in the study area. Those keys are only as good as the registrations behind them: abbreviations, spelling variants, house-number suffixes and wrong postcodes all split one building into several, or leave it unlocatable.

This notebook links the KBO building keys to **BeST Address**, the official Belgian address register, in four transparent stages:

| Stage | Rule | Risk |
|---|---|---|
| 1 · exact | same postcode, street and house number | none beyond the normalisation |
| 2 · base number | same postcode and street, house number without suffix (`12A` → `12`, `12-14` → `12`) | may merge neighbouring units |
| 3 · fuzzy street | same postcode, street name similar (Jaro-Winkler ≥ threshold, unique best candidate), number exact or base | wrong street if names are close |
| 4 · other postcode | same street and number in another study-area postcode, only if unique | district postcodes (2000/2018/2060…) are often mixed up |

Everything that does not match is classified by *why* it fails. The match rate per stage is the headline result: it says how far the address-based indicators in later notebooks can be trusted.

The link also adds two things the KBO data lacks: **coordinates**, and the **number of address units at the building** in BeST. The latter is a rough building-type proxy: twenty companies at a single-unit address is a different situation from twenty companies in an office building with forty units.

## 0 · Configuration

In [ ]:
from pathlib import Path
import hashlib, warnings

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT / "data" / "raw"
BEST_DIR = RAW_DIR / "best"                 # put the BeST Address CSV file(s) here
INTERIM = PROJECT / "data" / "interim"
PROCESSED = PROJECT / "data" / "processed"
BEST_DIR.mkdir(parents=True, exist_ok=True)

STUDY_POSTCODES = [str(p) for p in range(2000, 3000)] + ["9120", "9130"]
STREET_SIM = 0.92        # minimum Jaro-Winkler similarity for stage 3
STREET_SIM_MARGIN = 0.02 # best candidate must beat the runner-up by this much
PUBLIC_MODE = True

In [ ]:
def pseudo(value, prefix=""):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    return prefix + hashlib.sha256(str(value).encode()).hexdigest()[:10]

def show(df, sensitive_cols=(), n=20):
    out = df.head(n).copy()
    if PUBLIC_MODE:
        for c in sensitive_cols:
            if c in out.columns:
                out[c] = out[c].map(lambda v: pseudo(v, c[:3] + "_"))
    return out

# Same database as notebook 01: its tables and normalisation macros (norm_street, norm_number,
# norm_box) are reused, so KBO and BeST addresses go through exactly the same normalisation.
con = duckdb.connect(str(INTERIM / "kbo.duckdb"))
q = lambda sql: con.execute(sql).df()

missing = {"address_study", "address_summary"} - {r[0] for r in con.execute("SHOW TABLES").fetchall()}
assert not missing, f"Run notebook 01 first (missing tables: {missing})"

## 1 · Load BeST Address

BeST is published by BOSA from the three regional registers, one row per address (including box numbers). Different downloads use different column names: the NGI/geo.be CSV uses the official BeST field names (`housenumber`, `boxnumber`, `postal_info_objectid`, a `position` geometry in Lambert 2008), while derived files use `house_number`, `postcode` and WGS84 latitude/longitude. The cell below maps whichever variant is present to one standard view, and stops with a clear message if it can't.

In [ ]:
csvs = sorted(BEST_DIR.glob("*.csv"))
assert csvs, f"Put the BeST Address CSV(s) in {BEST_DIR}"
print([p.name for p in csvs])

con.execute(f"""
CREATE OR REPLACE TABLE best_raw AS
SELECT * FROM read_csv({[p.as_posix() for p in csvs]}, header=true, all_varchar=true,
                       union_by_name=true, filename=true)
""")

# Normalise header names: lower case, ':' and spaces to '_'
for (old,) in con.execute("SELECT column_name FROM (DESCRIBE best_raw)").fetchall():
    new = old.lower().replace(":", "_").replace(" ", "_")
    if old != new:
        con.execute(f'ALTER TABLE best_raw RENAME COLUMN "{old}" TO "{new}"')
present = {r[0] for r in con.execute("DESCRIBE best_raw").fetchall()}

# Candidate source columns per standard field, in order of preference
CANDIDATES = {
    "address_id":    ["best_id", "address_id", "fed_address_id", "id"],
    "postcode":      ["postcode", "postal_info_objectid", "postal_code", "zipcode"],
    "streetname_nl": ["streetname_nl"],
    "streetname_fr": ["streetname_fr"],
    "house_number":  ["housenumber", "house_number"],
    "box_number":    ["boxnumber", "box_number"],
    "status":        ["status"],
}
colmap = {}
for field, options in CANDIDATES.items():
    found = next((c for c in options if c in present), None)
    assert found, f"No column for '{field}' (tried {options}). Found: {sorted(present)}"
    colmap[field] = found

# Coordinates: WGS84 columns if present, otherwise a geometry column to convert
LATLON = next(((la, lo) for la, lo in [("epsg_4326_lat", "epsg_4326_lon"), ("lat", "lon"),
                                       ("latitude", "longitude")]
               if la in present and lo in present), None)
GEOM_COL = next((c for c in ["position", "geometry", "geom", "wkt"] if c in present), None)
SOURCE_CRS = "EPSG:3812"   # Lambert 2008, used by the NGI/geo.be download (note the _3812 in the file name)
assert LATLON or GEOM_COL, f"No coordinate columns found. Found: {sorted(present)}"
print("Column mapping:", colmap)
print("Coordinates from:", LATLON or f"{GEOM_COL} ({SOURCE_CRS})")

# BeST addresses are a public register, not personal data, so a sample can be shown
q(f"SELECT {', '.join(dict.fromkeys(list(colmap.values()) + ([GEOM_COL] if GEOM_COL else list(LATLON))))} "
  "FROM best_raw LIMIT 3")

In [ ]:
display(q(f"SELECT count(*) AS rows FROM best_raw"))
q(f"SELECT {colmap['status']} AS status, count(*) AS n FROM best_raw GROUP BY 1 ORDER BY 2 DESC")

Only addresses with status `current` are used: retired addresses would let outdated KBO registrations match, which hides exactly the stale data this notebook should expose.

Postcodes are taken as the last four digits of the postcode field, which works both for a plain postcode and for an identifier that ends in it. Coordinates given as a geometry are converted from Lambert 2008 to latitude/longitude with `pyproj`, and checked against a bounding box of Belgium.

In [ ]:
from pyproj import Transformer

STUDY_LIST = ",".join(f"'{p}'" for p in STUDY_POSTCODES)
c = colmap

if LATLON:
    coord_sql = f"try_cast({LATLON[0]} AS DOUBLE) AS lat, try_cast({LATLON[1]} AS DOUBLE) AS lon"
else:
    # Pull the first two numbers out of the geometry text, e.g. 'POINT (652345.1 713456.7)'
    coord_sql = (f"try_cast(regexp_extract_all({GEOM_COL}, '-?[0-9]+(?:\\.[0-9]+)?')[1] AS DOUBLE) AS x, "
                 f"try_cast(regexp_extract_all({GEOM_COL}, '-?[0-9]+(?:\\.[0-9]+)?')[2] AS DOUBLE) AS y")

best = q(f"""
SELECT {c['address_id']} AS address_id,
       regexp_extract({c['postcode']}, '([0-9]{{4}})$', 1)                      AS postcode,
       norm_street(coalesce(nullif({c['streetname_nl']}, ''), {c['streetname_fr']})) AS street_norm,
       norm_number({c['house_number']})                                         AS house_norm,
       regexp_extract(norm_number({c['house_number']}), '^[0-9]+')              AS base_number,
       norm_box({c['box_number']})                                              AS box_norm,
       {coord_sql}
FROM best_raw
WHERE regexp_extract({c['postcode']}, '([0-9]{{4}})$', 1) IN ({STUDY_LIST})
  AND lower({c['status']}) = 'current'
""")
assert len(best), ("No current BeST addresses in the study area. Check the status values and "
                   "the postcode column in the two cells above.")

if not LATLON:
    ok = best.x.notna() & best.y.notna()
    assert ok.mean() > 0.95, (f"Could not read coordinates from '{GEOM_COL}'. Sample: "
                             f"{con.execute(f'SELECT {GEOM_COL} FROM best_raw LIMIT 2').fetchall()}")
    tf = Transformer.from_crs(SOURCE_CRS, "EPSG:4326", always_xy=True)
    best["lon"], best["lat"] = tf.transform(best.x.values, best.y.values)
    best = best.drop(columns=["x", "y"])

in_belgium = best.lat.between(49.4, 51.6) & best.lon.between(2.5, 6.5)
print(f"{len(best):,} current addresses in the study area; {in_belgium.mean():.1%} with coordinates inside Belgium")
assert in_belgium.mean() > 0.95, "Coordinates fall outside Belgium: wrong SOURCE_CRS or swapped x/y?"

con.register("best_df", best)
con.execute("""
CREATE OR REPLACE TABLE best_addr AS SELECT * FROM best_df;

-- One row per building (all boxes together)
CREATE OR REPLACE TABLE best_building AS
SELECT postcode, street_norm, house_norm, any_value(base_number) AS base_number,
       avg(lat) AS lat, avg(lon) AS lon,
       count(*) AS n_units
FROM best_addr GROUP BY postcode, street_norm, house_norm;

-- One row per base number (for stage 2)
CREATE OR REPLACE TABLE best_base AS
SELECT postcode, street_norm, base_number,
       avg(lat) AS lat, avg(lon) AS lon, count(*) AS n_units
FROM best_addr WHERE base_number <> ''
GROUP BY postcode, street_norm, base_number;
""")
con.unregister("best_df")
q("""
SELECT (SELECT count(*) FROM best_addr) AS addresses,
       (SELECT count(*) FROM best_building) AS buildings,
       (SELECT count(DISTINCT postcode || street_norm) FROM best_building) AS streets
""")

## 2 · KBO buildings to match

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE kbo_building AS
SELECT DISTINCT s.building_key, s.postcode, s.street_norm, s.house_norm,
       regexp_extract(s.house_norm, '^[0-9]+') AS base_number,
       a.n_enterprises
FROM address_study s JOIN address_summary a USING (building_key);
""")
q("""
SELECT count(*) AS buildings, sum(n_enterprises) AS enterprise_links,
       sum((street_norm = '')::INT) AS no_street, sum((house_norm = '')::INT) AS no_number
FROM kbo_building
""")

## 3 · Matching in four stages

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE address_match (
  building_key VARCHAR, match_stage VARCHAR, best_street VARCHAR, street_similarity DOUBLE,
  lat DOUBLE, lon DOUBLE, n_best_units INTEGER);

-- Stage 1: exact
INSERT INTO address_match
SELECT k.building_key, '1_exact', b.street_norm, 1.0, b.lat, b.lon, b.n_units
FROM kbo_building k
JOIN best_building b USING (postcode, street_norm, house_norm);

-- Stage 2: base number
INSERT INTO address_match
SELECT k.building_key, '2_base_number', b.street_norm, 1.0, b.lat, b.lon, b.n_units
FROM kbo_building k
JOIN best_base b USING (postcode, street_norm, base_number)
WHERE k.building_key NOT IN (SELECT building_key FROM address_match);

-- Stage 3: fuzzy street within the same postcode, unique best candidate only
CREATE OR REPLACE TEMP TABLE street_candidates AS
WITH unmatched AS (
  SELECT DISTINCT postcode, street_norm FROM kbo_building
  WHERE street_norm <> '' AND building_key NOT IN (SELECT building_key FROM address_match)
),
best_streets AS (SELECT DISTINCT postcode, street_norm FROM best_building),
scored AS (
  SELECT u.postcode, u.street_norm AS kbo_street, b.street_norm AS best_street,
         jaro_winkler_similarity(u.street_norm, b.street_norm) AS sim
  FROM unmatched u JOIN best_streets b USING (postcode)
),
ranked AS (
  SELECT *, row_number() OVER w AS rk, lead(sim) OVER w AS runner_up
  FROM scored WINDOW w AS (PARTITION BY postcode, kbo_street ORDER BY sim DESC)
)
SELECT postcode, kbo_street, best_street, sim, runner_up
FROM ranked
WHERE rk = 1 AND sim >= {STREET_SIM} AND sim < 1
  AND (runner_up IS NULL OR sim - runner_up >= {STREET_SIM_MARGIN});

INSERT INTO address_match
SELECT k.building_key, '3_fuzzy_street', c.best_street, c.sim,
       coalesce(b.lat, bb.lat), coalesce(b.lon, bb.lon), coalesce(b.n_units, bb.n_units)
FROM kbo_building k
JOIN street_candidates c ON c.postcode = k.postcode AND c.kbo_street = k.street_norm
LEFT JOIN best_building b ON b.postcode = k.postcode AND b.street_norm = c.best_street AND b.house_norm = k.house_norm
LEFT JOIN best_base bb ON bb.postcode = k.postcode AND bb.street_norm = c.best_street AND bb.base_number = k.base_number
WHERE k.building_key NOT IN (SELECT building_key FROM address_match)
  AND coalesce(b.lat, bb.lat) IS NOT NULL;

-- Stage 4: same street and number in another study-area postcode, only if unique
INSERT INTO address_match
SELECT building_key, '4_other_postcode', best_street, 1.0, lat, lon, n_units FROM (
  SELECT k.building_key, b.street_norm AS best_street, b.lat, b.lon, b.n_units,
         count(*) OVER (PARTITION BY k.building_key) AS n_candidates
  FROM kbo_building k
  JOIN best_building b ON b.street_norm = k.street_norm AND b.house_norm = k.house_norm
                      AND b.postcode <> k.postcode
  WHERE k.street_norm <> '' AND k.house_norm <> ''
    AND k.building_key NOT IN (SELECT building_key FROM address_match)
) WHERE n_candidates = 1;
""")

dupes = con.execute("SELECT count(*) - count(DISTINCT building_key) FROM address_match").fetchone()[0]
assert dupes == 0, f"{dupes} buildings matched more than once"

### Match rate per stage

In [ ]:
rates = q("""
SELECT coalesce(m.match_stage, '5_unmatched') AS stage,
       count(*) AS buildings,
       sum(k.n_enterprises) AS enterprise_links
FROM kbo_building k LEFT JOIN address_match m USING (building_key)
GROUP BY 1 ORDER BY 1
""")
STAGES = ["1_exact", "2_base_number", "3_fuzzy_street", "4_other_postcode", "5_unmatched"]
rates = rates.set_index("stage").reindex(STAGES, fill_value=0).reset_index()
rates["share_buildings"] = (rates.buildings / rates.buildings.sum()).round(3)
rates["share_enterprises"] = (rates.enterprise_links / rates.enterprise_links.sum()).round(3)
rates["cumulative_buildings"] = (rates.buildings.cumsum() / rates.buildings.sum()).round(3)
rates

The enterprise-weighted share matters most for later scoring: a few unmatched concentrations weigh more than many unmatched single-company addresses.

### Checking the fuzzy matches

Stage 3 is the only stage that makes a judgement call. Read through these pairs (street names only, no house numbers) and adjust `STREET_SIM` if wrong matches appear. Record the decision in the README: this is the kind of choice that has to be documented.

In [ ]:
q("""
SELECT postcode, kbo_street, best_street, round(sim, 3) AS sim, round(runner_up, 3) AS runner_up
FROM street_candidates ORDER BY sim LIMIT 40
""")

### Why the rest does not match

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE address_unmatched AS
SELECT k.building_key, k.postcode, k.n_enterprises,
  CASE
    WHEN k.street_norm = '' THEN 'no street'
    WHEN k.house_norm = ''  THEN 'no house number'
    WHEN NOT EXISTS (SELECT 1 FROM best_building b
                     WHERE b.postcode = k.postcode AND b.street_norm = k.street_norm)
     AND NOT EXISTS (SELECT 1 FROM street_candidates c
                     WHERE c.postcode = k.postcode AND c.kbo_street = k.street_norm)
         THEN 'street not found in postcode'
    ELSE 'number not found on street'
  END AS reason
FROM kbo_building k
WHERE k.building_key NOT IN (SELECT building_key FROM address_match);
""")
display(q("""
SELECT reason, count(*) AS buildings, sum(n_enterprises) AS enterprise_links
FROM address_unmatched GROUP BY 1 ORDER BY 2 DESC
"""))
# Unmatched streets: street names are not personal data, so they can be shown
q("""
SELECT k.postcode, k.street_norm, count(*) AS buildings, sum(k.n_enterprises) AS enterprise_links
FROM address_unmatched u JOIN kbo_building k USING (building_key)
WHERE u.reason = 'street not found in postcode'
GROUP BY 1, 2 ORDER BY enterprise_links DESC LIMIT 25
""")

Typical causes to look for in this list: streets renamed since registration, abbreviations the normaliser does not know yet (add them to `norm_street` in notebook 01 and rerun both notebooks), French street names in a Dutch-language municipality, and postcodes of the wrong district.

## 4 · Building type proxy

BeST counts every box as an address unit. Cross-tabulating units with the number of KBO enterprises gives a first view of where concentrations sit.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE address_summary_geo AS
SELECT a.*, m.match_stage, m.lat, m.lon, m.n_best_units,
       CASE WHEN m.n_best_units IS NULL THEN 'unknown'
            WHEN m.n_best_units = 1 THEN '1 unit'
            WHEN m.n_best_units <= 5 THEN '2-5 units'
            WHEN m.n_best_units <= 20 THEN '6-20 units'
            ELSE '>20 units' END AS building_size
FROM address_summary a LEFT JOIN address_match m USING (building_key);
""")

xt = q("""
SELECT building_size,
       CASE WHEN n_enterprises = 1 THEN '1'
            WHEN n_enterprises <= 4 THEN '2-4'
            WHEN n_enterprises <= 9 THEN '5-9'
            ELSE '10+' END AS enterprises,
       count(*) AS buildings
FROM address_summary_geo GROUP BY 1, 2
""").pivot(index="building_size", columns="enterprises", values="buildings").fillna(0).astype(int)
xt = xt.reindex(index=["1 unit", "2-5 units", "6-20 units", ">20 units", "unknown"],
                columns=["1", "2-4", "5-9", "10+"]).fillna(0).astype(int)
xt

The top-right of the upper rows (many enterprises, few units) is where notebook 03 will start looking. It is not a finding yet: a single-unit building can be a large accountancy office, and BeST units are registry units, not floor space.

### Where the concentrations are

Aggregated hexbin map: buildings with 5 or more enterprises, coloured by count per hexagon. Aggregation keeps individual addresses out of the published figure.

In [ ]:
pts = q("""
SELECT lon, lat FROM address_summary_geo
WHERE lat IS NOT NULL AND n_enterprises >= 5
""")

fig, ax = plt.subplots(figsize=(7, 7))
if len(pts):
    hb = ax.hexbin(pts.lon, pts.lat, gridsize=40, mincnt=1, cmap="Blues", linewidths=0.2)
    cb = fig.colorbar(hb, ax=ax, shrink=0.7)
    cb.set_label("Buildings with 5+ enterprises", color="#444")
    cb.outline.set_visible(False)
    cb.locator = MaxNLocator(integer=True); cb.update_ticks()
    cb.ax.tick_params(colors="#666", labelsize=8)
ax.set_aspect(1 / 0.62)       # rough correction for longitude at ~51°N
ax.set_xlabel("Longitude", color="#444")
ax.set_ylabel("Latitude", color="#444")
ax.set_title("Company concentrations in the study area", loc="left", color="#222")
for s in ax.spines.values():
    s.set_visible(False)
ax.tick_params(colors="#666", labelsize=8)

# Licence art. 2.8: state source and update date of the KBO data with every reuse
meta = dict(con.execute("SELECT Variable, Value FROM raw_meta").fetchall())
fig.text(0.01, 0.005,
         f"Bron: KBO Open Data, FOD Economie (extract {meta.get('ExtractNumber', '?')}, "
         f"stand {meta.get('SnapshotDate', '?')}); adressen: BeST Address (BOSA). Bewerking: eigen analyse.",
         fontsize=7, color="#666")
plt.show()

## 5 · Write outputs

In [ ]:
for name in ["address_match", "address_unmatched", "address_summary_geo"]:
    con.execute(f"COPY {name} TO '{(PROCESSED / (name + '.parquet')).as_posix()}' (FORMAT parquet)")
sorted(p.name for p in PROCESSED.glob("address_*.parquet"))

## Notes for the README

- Report the final match rate (buildings and enterprise-weighted) and the share per stage.
- Report the `STREET_SIM` threshold and what the manual check of the fuzzy pairs showed.
- State the main causes of non-matches and what was done about them.

Next: **03 · Scoring**, using `address_summary_geo` (concentration, recent starts, sector mix, building size) and the network components from notebook 01.